# 11주차 · 시계열 예측

**⏱ 60분** · 미국 월별 항공 여객 1990년 1월 \~ 2004년 4월

## 🎯 이번 주의 질문

2001년 9월 미국의 항공 여객이 갑자기 크게 줄었다. 항공사와 정부가 감편·지원 규모를 정하려면 얼마나 줄었는지를 숫자로 알아야 한다. 그런데 2001년 9월 전과 후를 그대로 비교하면 안 된다. 항공 여객은 해마다 늘고 있었고, 달마다 크게 오르내리기 때문이다. 그래서 미국 교통통계국(BTS)은 2001년 8월까지의 기록으로 **이전 추세가 이어졌을 때의 값(기준선)을 예측**하고, 실제 값과의 차이를 그 감소의 크기로 보았다. 이 기준선을 만드는 일이 시간 순서로 쌓인 과거 기록으로 앞으로의 값을 내는 **시계열 예측(time series forecasting)** 이다.

> **2001년 9월 이후 미국 항공 여객은 이전 추세로 예측한 기준선보다 얼마나 적었고, 그 차이는 얼마나 이어졌나?**

**데이터:** BTS 보고서 *Estimated Impacts of September 11th on US Travel*의 월별 교통량(Shmueli 외 교과서 연습문제 자료) 중 항공 여객을 쓴다.

| 셀 표시 | 할 일 |
|---|---|
| `# ✍️ 직접 입력` | 위에 보이는 코드를 직접 쳐서 실행한다 |
| `# ▶ 그대로 실행` | 준비·그림 코드이다. 실행만 한다 |

> **실습본이다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누른다. 사본을 저장하지 않으면 입력한 코드가 사라진다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행한다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑해 실행한다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행. 미국 월별 교통량을 읽음
import os
import sys
for _p in (".", "..", "../.."):
    if os.path.exists(os.path.join(_p, "balab.py")):
        sys.path.insert(0, _p); break
else:
    !wget -q https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py

from balab import load
df = load("sept11_travel")
print("준비 완료!", df.shape)

---
## 1. 자료의 열과 월별 시계열

시계열은 일정한 간격으로 기록한 값을 시간 순서대로 놓은 것이다. 표의 앞부분부터 본다.

```python
df.head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 한 행이 한 달이다. 각 열의 의미는 다음과 같다.

| 열 | 뜻 | 단위 |
|---|---|---|
| `Month` | 기록한 달. `Jan-90`은 1990년 1월 | 월 이름-연도 두 자리 글자 |
| `Air RPM (000s)` | 항공 유상 여객마일(revenue passenger miles). 돈을 낸 승객 1명을 1마일 태우면 1이다. 승객 수와 이동 거리를 함께 담은 항공 수요의 표준 지표이다 | 천 여객마일 |
| `Rail PM` | 철도 여객마일(passenger miles). 승객 1명을 1마일 태우면 1이다 | 여객마일 |
| `VMT (billions)` | 자동차 주행거리(vehicle miles traveled). 차 1대가 1마일 달리면 1이다 | 십억 마일 |

해당 실습에서는 `Air RPM (000s)`를 쓴다.

우선, `Month`를 날짜로 바꿔야 한다. 글자로는 날짜 계산이나 달 꺼내기를 할 수 없다. 다음 코드와 같이 바꾼다.

```python
import pandas as pd  # 표를 다루는 pandas

df["Month"] = pd.to_datetime(df["Month"], format="%b-%y")  # 글자의 형식 %b-%y: 월 이름-연도 두 자리
df = df.set_index("Month")  # 날짜를 행 이름(index)으로 둠
print(df.index.min(), df.index.max(), len(df))
```

In [ ]:
# ✍️ 위 코드를 입력하세요


다음으로, `Air RPM (000s)`의 단위를 관리가 가능하도록 100만으로 나눈다. 원래 값이 천 여객마일 단위라, 100만으로 나누면 십억 여객마일 단위가 된다.

```python
df["air"] = df["Air RPM (000s)"] / 1e6  # 천 여객마일 ÷ 100만 = 십억 여객마일
df[["Air RPM (000s)", "air"]].head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


In [ ]:
# ▶ 그대로 실행 — 월별 항공 여객, 빨간 점선이 2001년 9월
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 3.5))
plt.plot(df.index, df["air"])
plt.axvline(pd.Timestamp("2001-09-01"), color="red", linestyle="--")
plt.xlabel("Month")
plt.ylabel("Air RPM (billions)")
plt.show()

💬 **결과 해설**
- 항공 여객은 1990년대 내내 꾸준히 늘었다. 긴 기간에 걸친 이 흐름이 **추세**이다.
- 해마다 같은 모양의 오르내림이 되풀이된다. 일정한 주기로 되풀이되는 패턴이 **계절성**이다.
- 2001년 8월 69.0에서 9월 39.1로 떨어졌다. 그런데 9월은 원래 8월보다 적은 달이고, 해마다 여객이 늘던 중이었다. 이 낙폭이 평소의 계절 변동을 넘는 감소인지는 그래프만으로 알 수 없다. "이전 추세가 이어졌다면 9월은 얼마였을까"가 필요하다.

---
## 2. 기간 나누기

기준선은 2001년 9월 이후의 변화가 들어 있지 않은 **2001년 8월까지의 140개월**로 만들기로 한다. 다음 코드와 같이 기간을 2001년 8월까지와 2001년 9월부터로 나눈다.

```python
pre = df.loc[:"2001-08"]  # 2001년 8월까지
post = df.loc["2001-09":]  # 2001년 9월부터
print(len(pre), len(post))
```

In [ ]:
# ✍️ 위 코드를 입력하세요


다음으로, 2001년 8월까지의 140개월에서 **마지막 12개월**(2000년 9월 \~ 2001년 8월)을 테스트 기간으로 떼어 두기로 한다. 기준선을 만들 방법은 여러 가지이고, 어느 방법이 앞을 잘 맞히는지는 2001년 8월까지의 기록에서 앞을 맞혀 보고 판단한다. 방법은 그 앞 128개월인 학습 기간으로만 맞춘다. 질문의 중심이 2001년 9월 이후 첫 1년이므로 1년 앞을 맞히는 능력으로 고른다. 미래는 과거 뒤에 오므로 다음 코드와 같이 시간 순서대로 자른다.

```python
train = pre.iloc[:-12]  # 학습 기간: 마지막 12개월 앞까지
test = pre.iloc[-12:]  # 테스트 기간: 마지막 12개월
y_test = test["air"]  # 테스트 기간의 실제 값
print(train.index.max(), test.index.min(), len(train), len(test))
```

In [ ]:
# ✍️ 위 코드를 입력하세요


---
## 3. naive 예측과 seasonal naive 예측

이제 기준선을 만들 방법을 찾는다. 후보 방법마다 학습 기간으로 모형을 맞추고 테스트 기간 12개월을 예측한 뒤, 예측이 실제 값과 얼마나 가까운지 먼저 그림으로 본다. 모형이 학습 기간 안에서 계산한 값은 예측과 구별해 **적합값**이라 부른다. 방법이 여섯 개이므로 예측값과 적합값을 방법 이름별로 모아 두고, 그림도 함수 하나로 그린다.

In [ ]:
# ▶ 그대로 실행 — 예측값·적합값을 모을 곳과 그림 함수
# 방법 이름으로 꺼낼 수 있게 딕셔너리에 모음
preds = {}  # 방법 이름: 테스트 기간의 예측값
fits = {}  # 방법 이름: 학습 기간의 적합값


# 실제 값(회색 점), 적합값(실선), 테스트 기간의 예측(점선)을 1997년부터 그림. 세로선이 테스트 기간의 시작
def plot_forecast(name):
    plt.figure(figsize=(9, 3.5))
    plt.plot(pre.loc["1997":].index, pre.loc["1997":, "air"], "o", color="gray", markersize=3, label="Actual")
    if name in fits:
        plt.plot(fits[name].loc["1997":].index, fits[name].loc["1997":], color="C0", label="Fitted (training)")
    plt.plot(preds[name].index, preds[name], "--o", color="C0", markersize=4, label="Forecast (test)")
    plt.axvline(test.index[0], color="black", linewidth=0.8)
    plt.title(name)
    plt.ylabel("Air RPM (billions)")
    plt.legend()
    plt.show()

가장 단순한 방법부터 본다. **naive 예측**은 마지막으로 관측한 값이 앞으로도 그대로 이어진다고 보고, 학습 기간의 마지막 값인 2000년 8월의 값을 테스트 기간 12개월 내내 쓴다.

```python
# 값 하나를 테스트 기간의 모든 달에 넣은 Series, 행 이름이 테스트 기간의 날짜라 실제 값과 달끼리 비교됨
preds["naive"] = pd.Series(train["air"].iloc[-1], index=test.index)
preds["naive"].round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


In [ ]:
# ▶ 그대로 실행 — naive 예측 그림
plot_forecast("naive")

💬 성수기인 8월의 값을 가을·겨울에도 그대로 써서, 9월 이후 실제 값보다 크게 위에 있다.

naive 예측이 크게 빗나간 것은 달마다 오르내리는 계절성을 무시했기 때문이다. **seasonal naive 예측**은 계절성을 반영해, 한 주기인 12개월 전 같은 달의 값을 그 달의 예측으로 쓴다. 테스트 기간이 12개월이므로 학습 기간의 마지막 12개월 값이 그대로 테스트 기간의 예측이 된다.

```python
# iloc[-12:]: 마지막 12개월, tolist(): 원래 날짜를 떼고 값만 꺼냄(Series를 그대로 넘기면 원래 날짜에 맞춰 값을 찾음)
preds["seasonal naive"] = pd.Series(train["air"].iloc[-12:].tolist(), index=test.index)
preds["seasonal naive"].round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


In [ ]:
# ▶ 그대로 실행 — seasonal naive 예측 그림
plot_forecast("seasonal naive")

💬 계절 모양은 따라가지만, 해마다 여객이 늘어나는 만큼은 반영하지 못해 테스트 기간에는 대체로 실제보다 조금 아래에 있다.

---
## 4. 자기회귀 모형

**자기회귀(autoregression)** 는 직전 N개월의 값을 입력하여 다음값을 출력하는 방법이다. 따라서 $y_{t-4}, \dots, y_{t-1}$이 입력값에 $y_t$값이 출력값이 되도록 데이터를 다시 재구조화 해야한다. 지금 표에는 `air` 열만 있으므로, pandas의 `shift`로 이 열을 아래로 밀어 과거 값의 열을 만든다. 한 칸 밀면 각 행에 지난달의 값이 오고, 열두 칸 밀면 1년 전 같은 달의 값이 온다. AR(4)에는 한 칸부터 네 칸까지 민 열 네 개를, 계절 자기회귀에는 열두 칸 민 열 하나를 쓴다.

```python
# shift(k): k칸 앞, 곧 k개월 전의 값
df["lag1"] = df["air"].shift(1)
df["lag2"] = df["air"].shift(2)
df["lag3"] = df["air"].shift(3)
df["lag4"] = df["air"].shift(4)
df["lag12"] = df["air"].shift(12)
df[["air", "lag1", "lag2", "lag3", "lag4", "lag12"]].head(13).round(2)  # lag12가 처음 채워지는 1991년 1월까지
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 앞쪽 행은 직전 값이 없어 비어 있다(`NaN`). `lag12`는 1991년 1월부터 채워진다.

### 자기회귀 AR(4)

먼저 직전 4개월의 값으로 그 달의 값을 예측하는 AR(4)를 맞춘다. 앞에서 만든 lag 열 네 개가 입력이고 `air`가 출력이다. 학습 기간에서 입력과 출력을 꺼내되, 처음 4개월은 직전 값이 없어 입력이 비어 있으므로 빼고 맞춘다.

```python
lags = ["lag1", "lag2", "lag3", "lag4"]
# lag 열은 df에 붙였으므로 학습 기간의 행을 df.loc[train.index]로 꺼냄, 입력이 모자란 처음 4개월은 iloc[4:]로 뺌
X_train = df.loc[train.index, lags].iloc[4:]
y_train = train["air"].iloc[4:]
X_train.shape
```

In [ ]:
# ✍️ 위 코드를 입력하세요


이 입력과 출력으로 회귀 계수를 구한다. 4주차와 같은 최소제곱 회귀인데, 여기서는 맞춘 모형에 새 입력을 넣어 예측값을 바로 낼 수 있는 scikit-learn의 `LinearRegression`을 쓴다. 맞춘 모형으로 학습 기간의 적합값도 구해 둔다.

```python
from sklearn.linear_model import LinearRegression  # 회귀 모형

model = LinearRegression().fit(X_train, y_train)  # fit: 계수를 구함
# predict: 입력을 넣으면 값을 냄, 학습 기간의 입력을 넣은 값이 적합값
fits["AR(4)"] = pd.Series(model.predict(X_train), index=X_train.index)
print(model.intercept_)
pd.Series(model.coef_, index=lags).round(3)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 직전 달의 값에 가장 큰 계수가 붙었다.

이제 AR(4)로 테스트 기간 12개월을 예측한다. AR(4)는 직전 4개월의 값을 알아야 다음 달을 예측할 수 있다. 2000년 9월은 직전 4개월인 2000년 5월부터 8월까지가 모두 학습 기간에 있어 바로 예측할 수 있다. 그런데 10월을 예측하려면 9월의 값이 필요하고, 2000년 8월 말에 예측하는 상황이라 9월의 실제 값은 아직 알 수 없다. 그래서 9월의 실제 값 대신 방금 낸 9월의 예측값을 넣어 10월을 예측하고, 같은 방식으로 한 달씩 이어 12개월을 예측한다. 이렇게 예측값을 다시 입력으로 써서 여러 달 앞까지 예측하는 것을 **다단계 예측**이라 한다.

```python
history = list(train["air"])  # 학습 기간의 값, 한 달을 예측할 때마다 그 예측값을 뒤에 붙임
y_pred = []
for month in test.index:  # 테스트 기간의 달을 차례로
    # history 맨 뒤 4개 값(직전 4개월)으로 입력을 만듦, history[-1]이 직전 달
    X_next = pd.DataFrame([[history[-1], history[-2], history[-3], history[-4]]], columns=lags)
    next_value = model.predict(X_next)[0]
    y_pred.append(next_value)
    history.append(next_value)  # 이 예측값이 다음 달의 직전 달 값이 됨
preds["AR(4)"] = pd.Series(y_pred, index=test.index)
preds["AR(4)"].round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 10월의 입력 `lag1`에는 9월의 예측값 64.91이 들어갔다.

In [ ]:
# ▶ 그대로 실행 — 자기회귀 AR(4) 그림
plot_forecast("AR(4)")

💬 학습 기간의 적합값은 실제 값을 한 달 늦게 따라간다. 테스트 기간의 예측은 8월의 높은 값에서 출발해 달마다 조금씩 낮아질 뿐, 계절 모양을 그리지 않는다. 입력이 직전 4개월뿐이라 1년 전 같은 달의 값이 들어 있지 않기 때문이다.

### 계절 자기회귀

**계절 자기회귀**는 1년 전 같은 달의 값 `lag12` 하나를 입력으로 쓰는 자기회귀다. AR(4)의 입력에 없던 계절 정보를 넣는 셈이다. 맞추는 방법은 AR(4)와 같고, `lag12`가 비어 있는 처음 12개월을 빼고 맞춘다.

```python
X_train = df.loc[train.index, ["lag12"]].iloc[12:]  # 입력이 없는 처음 12개월은 뺌
y_train = train["air"].iloc[12:]
model = LinearRegression().fit(X_train, y_train)
fits["seasonal AR"] = pd.Series(model.predict(X_train), index=X_train.index)
print(model.intercept_, model.coef_)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 1년 전 같은 달의 값에 1.055를 곱하고 0.56을 뺀다. 계수가 1보다 조금 커서, 해마다 늘어나는 만큼을 담는다.

이제 테스트 기간을 예측한다. 계절 자기회귀는 AR(4)와 달리 다단계 예측이 필요 없다. 테스트 기간 각 달의 입력은 12개월 전, 곧 1999년 9월부터 2000년 8월까지의 값이고, 모두 학습 기간에 있는 실제 값이기 때문이다. 그래서 12개월을 한 번에 예측한다.

```python
X_test = df.loc[test.index, ["lag12"]]  # 테스트 기간 행의 lag12, 모두 학습 기간의 실제 값
preds["seasonal AR"] = pd.Series(model.predict(X_test), index=test.index)
preds["seasonal AR"].round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


In [ ]:
# ▶ 그대로 실행 — 계절 자기회귀 그림
plot_forecast("seasonal AR")

💬 seasonal naive 예측과 같은 계절 모양을 조금 키운 값이라, 테스트 기간에는 대체로 실제보다 조금 위에 있다.

---
## 5. 추세·계절성 회귀

자기회귀는 과거의 값으로 다음 값을 예측했다. **추세·계절성 회귀**는 과거의 값 대신 시간 자체를 설명 변수로 쓰는 회귀 모형이다. 해마다 늘어나는 흐름은 시점 t로, 달마다 오르내리는 모양은 그 행이 몇 월인지를 나타내는 변수로 설명한다. 이를 위해 두 종류의 열을 만든다.
- **t:** 시점 번호. 1990년 1월이 1, 2004년 4월이 172이다. 2001년 9월 이후 기간에도 번호를 이어서 매긴다.
- **계절성 dummy 변수:** 그 행이 해당 달이면 1, 아니면 0인 열. 기준 달(1월)을 빼고 2월부터 12월까지 11개를 만든다. 12개를 모두 넣으면 어느 행이든 12개 열의 합이 1이 되어 상수항과 겹치기 때문이다. 열 이름은 `m_2`(2월이면 1)부터 `m_12`(12월이면 1)까지이다.

```python
import numpy as np  # 배열을 다루는 numpy

df["t"] = np.arange(1, len(df) + 1)  # 1부터 172까지의 시점 번호
# 달마다 0·1 열을 만듦(열 이름 m_2 ~ m_12), drop_first=True: 1월 열을 뺌
dummies = pd.get_dummies(df.index.month, prefix="m", drop_first=True).astype(int)
dummies.index = df.index
df = pd.concat([df, dummies], axis=1)  # 표 오른쪽에 붙임
df[["t"] + list(dummies.columns)].head(12)  # 새로 만든 열 전부, 1990년 1월부터 12월까지
```

In [ ]:
# ✍️ 위 코드를 입력하세요


두 모형 모두 학습 기간 128개월 전체로 맞춘다.

### 선형 추세 모형

먼저 추세만 담는 모형이다. 시점 번호 t 하나로 직선을 맞추며, 식은 여객 = β₀ + β₁ × t이다. 4주차 회귀와 같고, 설명 변수가 시점 번호일 뿐이다.

```python
X_train = df.loc[train.index, ["t"]]
y_train = train["air"]
model = LinearRegression().fit(X_train, y_train)
fits["linear"] = pd.Series(model.predict(X_train), index=X_train.index)
print(model.intercept_, model.coef_)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 한 달에 0.176(십억 여객마일)씩, 1년에 약 2.1씩 늘어나는 직선이다.

이 직선으로 테스트 기간을 예측한다. t는 앞으로 올 달에도 번호를 이어 매겨 미리 알 수 있으므로, 자기회귀처럼 예측값을 입력으로 이어 쓸 필요 없이 12개월을 한 번에 예측한다.

```python
X_test = df.loc[test.index, ["t"]]  # 테스트 기간의 t(129~140)
preds["linear"] = pd.Series(model.predict(X_test), index=test.index)
preds["linear"].round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


In [ ]:
# ▶ 그대로 실행 — 선형 추세 모형 그림
plot_forecast("linear")

💬 직선은 늘어나는 흐름을 따라가지만, 해마다 되풀이되는 오르내림은 담지 않는다. 실제 값이 직선 위아래로 크게 오르내린다.

### 선형 추세 + 계절성 모형

선형 추세 모형에 계절성 dummy 변수 11개를 설명 변수로 더한다. 추세를 나타내는 직선 위에, 달마다 기준인 1월과의 차이만큼 더하고 빼는 모형이다.

```python
cols = ["t"] + list(dummies.columns)  # t와 m_2 ~ m_12, 학습·예측·기준선에서 같은 열 목록을 씀
X_train = df.loc[train.index, cols]
model = LinearRegression().fit(X_train, y_train)
fits["linear + season"] = pd.Series(model.predict(X_train), index=X_train.index)
print(model.intercept_)
pd.Series(model.coef_, index=cols).round(3)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설** — 4주차처럼 다른 설명 변수가 같을 때의 차이로 읽는다.
- `t` = 0.171: 같은 달끼리 보면 한 달에 0.171씩 늘어난다.
- `m_8` = 13.34: 같은 시점의 추세에서 8월은 기준인 1월보다 13.34(십억 여객마일) 많다. `m_2` = −2.52: 2월은 1월보다 적다.
- 계절성 dummy 변수는 해마다 같은 크기만큼 더하고 뺀다. 계절 차이가 해마다 같은 크기라고 가정한 것이다.

```python
X_test = df.loc[test.index, cols]
preds["linear + season"] = pd.Series(model.predict(X_test), index=test.index)
preds["linear + season"].round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


In [ ]:
# ▶ 그대로 실행 — 선형 추세 + 계절성 모형 그림
plot_forecast("linear + season")

💬 적합값이 계절 모양까지 따라가고, 테스트 기간의 예측도 실제 값 가까이에 있다.

---
## 6. 테스트 기간의 오차로 고르기

여섯 방법의 예측을 그림으로 보았으니, 이제 테스트 기간에 얼마나 틀렸는지를 숫자로 비교해 기준선을 만들 방법을 고른다. 달마다 실제 값에서 예측값을 뺀 값이 **예측 오차**이고, 양수면 실제보다 적게 예측한 것이다. seasonal naive 예측의 오차부터 본다.

```python
e = y_test - preds["seasonal naive"]  # 행 이름(날짜)이 같아 같은 달끼리 빼짐
e.round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


12개의 오차를 하나씩 비교하기는 어려우므로 숫자 하나로 요약한다.

| 지표 | 계산 | 뜻 |
|---|---|---|
| MSE | 오차 제곱의 평균 | 오차의 크기, 큰 오차에 더 큰 벌점 |
| MAPE | (오차 ÷ 실제 값)의 절댓값 평균 × 100 | 실제 값의 몇 % 틀리는지 |

같은 계산을 여섯 방법에 되풀이하므로 두 지표의 계산을 함수로 만들어 둔다.

```python
def errors(y_true, y_pred):
    e = y_true - y_pred
    return pd.Series({"MSE": (e ** 2).mean(),
                      "MAPE": (e / y_true).abs().mean() * 100})

errors(y_test, preds["seasonal naive"]).round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 실제 값의 약 2.7% 틀렸다.

같은 계산을 여섯 방법 모두에 해서 한 표로 모으고, MAPE가 작은 순서로 놓는다.

```python
# {name: ... for name in preds}: 방법 이름마다 errors를 계산해 모음
table = pd.DataFrame({name: errors(y_test, preds[name]) for name in preds}).T
table.sort_values("MAPE").round(2)  # MAPE가 작은 순서로
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 테스트 MAPE가 선형 추세 + 계절성 모형 **1.88**, seasonal naive 예측 2.74, 계절 자기회귀 2.76, 선형 추세 모형 6.87, 자기회귀 AR(4) 12.80, naive 예측 14.22이면 맞다.

💬 **결과 해설**
- 테스트 MAPE가 가장 작은 방법을 고른다. 선형 추세 + 계절성 모형이 1.88%로 가장 작다.
- 1년 전 같은 달의 값이나 계절성 dummy 변수가 없는 방법(naive 예측 14.22%, AR(4) 12.80%, 선형 추세 모형 6.87%)은 크게 틀린다.

방법을 학습 기간의 오차가 아니라 테스트 기간의 오차로 고르는 이유를 숫자로 본다. 적합값이 있는 네 방법의 학습 오차와 테스트 오차를 나란히 놓는다. 자기회귀의 학습 오차는 입력이 있는 달, 곧 AR(4)는 5번째 달부터, 계절 자기회귀는 13번째 달부터 계산한다.

```python
# train: 적합값으로 계산한 학습 오차, test: 예측값으로 계산한 테스트 오차
mape = pd.DataFrame({"train": [errors(train["air"], fits[name])["MAPE"] for name in fits],
                     "test": [errors(y_test, preds[name])["MAPE"] for name in fits]},
                    index=list(fits))
mape.round(2)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **결과 해설**
- AR(4)는 학습 MAPE가 7.50%인데 테스트 MAPE는 12.80%로 커진다. 학습 오차는 모형이 이미 본 자료로 계산하므로, 방법은 테스트 오차로 고른다.
- 기준선은 테스트 MAPE가 가장 작은 **선형 추세 + 계절성 모형**으로 만든다.

---
## 7. 2001년 9월 이후의 기준선

이제 고른 선형 추세 + 계절성 모형으로 기준선을 만든다. 방법을 고를 때는 테스트 기간 12개월을 떼어 두었지만, 기준선은 그 12개월까지 포함한 2001년 8월까지의 140개월 전체로 다시 맞춘다. 2001년 9월 바로 앞의 흐름까지 기준선에 담기 위해서이다. 다시 맞춘 식에 2001년 9월 이후 32개월의 t와 달을 넣은 값이 이전 추세가 이어졌을 때의 예측, 곧 **기준선**이다. t와 달은 미래에도 미리 알 수 있으므로 32개월을 한 번에 예측한다.

```python
# 5절과 같은 모형, 맞추는 행은 pre(2001년 8월까지 140개월), 예측하는 행은 post
X = df.loc[pre.index, cols]
y = pre["air"]
model = LinearRegression().fit(X, y)
baseline = pd.Series(model.predict(df.loc[post.index, cols]), index=post.index)
pd.DataFrame({"actual": post["air"], "baseline": baseline}).round(1)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


In [ ]:
# ▶ 그대로 실행 — 실제 값과 기준선
plt.figure(figsize=(9, 3.5))
plt.plot(df.loc["1998":].index, df.loc["1998":, "air"], color="gray", label="Actual")
plt.plot(baseline.index, baseline, linestyle="--", label="Baseline")
plt.axvline(pd.Timestamp("2001-09-01"), color="red", linestyle="--")
plt.ylabel("Air RPM (billions)")
plt.legend()
plt.show()

💬 2001년 9월 이후 실제 값(회색)이 기준선(점선)보다 계속 아래에 있다. 두 선 사이가 이전 추세에서 벗어난 부분이다.

---
## 8. 얼마나 줄었고, 언제까지 이어졌나

실제 값이 기준선보다 적은 만큼이 2001년 9월 이후의 감소이다. 달마다 여객 규모가 다르므로, 실제 값과 기준선의 차이를 기준선 대비 비율(%)로 나타낸다.

```python
gap_pct = (post["air"] - baseline) / baseline * 100  # 실제 값이 기준선보다 몇 % 적은지, 같은 날짜끼리 계산됨
gap_pct.round(1)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 2001년 9월 **−33.3**, 2004년 4월 **−4.5**이면 맞다.

💬 2001년 9월 한 달 동안 항공 여객은 기준선보다 3분의 1이 적었다. 차이는 달마다 줄어들었지만 2년 반이 지난 2004년 4월에도 4.5% 적다.

### 결정

> **2001년 9월 미국 항공 여객은 이전 추세로 예측한 기준선보다 33% 적었다. 차이는 점점 줄었지만 2년 반이 지난 2004년 4월에도 4.5% 적어, 감소는 일시적이 아니라 계속됐다.**

기준선도 예측이므로 테스트 기간 정도(MAPE 2% 안팎)의 오차는 있다. 2001년 9월의 33% 감소는 그보다 훨씬 크다.

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 자료의 열 | `Air RPM (000s)` 항공 유상 여객마일(천), `Rail PM` 철도 여객마일, `VMT (billions)` 자동차 주행거리 |
| 시계열 만들기 | 날짜 글자 → `pd.to_datetime` → index. 172개월, 늘어나는 추세와 12개월 계절성 |
| 기간 나누기 | 기준선은 2001년 8월까지의 140개월로만, 그 마지막 12개월이 테스트 기간 |
| naive·seasonal naive 예측 | 마지막 값 / 마지막 12개월을 그대로 씀 |
| 자기회귀 | `shift`로 lag 열. AR(4)는 직전 4개월을 입력으로 다단계 예측, 계절 자기회귀는 `lag12` 하나 |
| 추세·계절성 회귀 | t, 계절성 dummy 변수 11개(1월 기준). 2001년 9월 이후 행의 열도 미리 알 수 있음 |
| 오차 지표 | MSE는 오차 제곱의 평균, MAPE는 실제 값의 몇 % 틀리는지 |
| 비교 | 테스트 MAPE: 선형 추세 + 계절성 모형 1.88 · seasonal naive 예측 2.74 · 계절 자기회귀 2.76 · 선형 추세 모형 6.87 · 자기회귀 AR(4) 12.80 · naive 예측 14.22 |
| 학습 오차와 테스트 오차 | AR(4)는 학습 7.50% → 테스트 12.80%, 방법은 테스트 오차로 고름 |
| 기준선과의 차이 | 2001-09 −33.3%, 2004-04에도 −4.5% |

**랩:** 관광지식정보시스템에서 국적별 입국 월별 자료를 직접 내려받아, 배정된 나라의 2019년을 이 실습의 방법으로 예측하고 비교한다.